# 02 — Limpieza y Transformación INE

### Decisiones de diseño (documentadas en `01_ine.ipynb`)
| Decisión | Elección | Motivo |
|----------|----------|--------|
| Nivel geográfico | **Municipio** → agregado a **Provincia** | TFG usa provincia como unidad |
| Método agregación | **Mediana** | Robusta a outliers (municipios pequeños extremos) |
| Separador decimal | Normalizar a **punto (.)** | Mixto en fuente: `1_renta` usa `.`, resto usa `,` |
| Valores faltantes | `NaN` (ya capturados en lectura) | INE usa `..` = dato no disponible |
| Imputación NaN | **Sin imputar** — columna `_flag` indica ausencia | Preservar integridad |
| Clave join | `cod_provincia` + `year` | Compatible con EVR y otros dominios |

In [8]:
#=====================================================================
# CELDA 1: Carga PATH
#=====================================================================


# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Estilo visual ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── Rutas (auto-discovery: sube hasta encontrar /data) ────────────────────────
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR   = PROJECT_ROOT / 'data' / 'renta_ine'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'clean'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('DATA existe  :', DATA_DIR.exists())

# ── Ficheros ──────────────────────────────────────────────────────────────────
FILES = {
    'renta':               DATA_DIR / '1.Indicadores de renta media y mediana.csv',
    'fuente_ing':          DATA_DIR / '2.Distribución por fuente de ingresos.csv',
    'umbrales_edad':       DATA_DIR / '4.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y tramos de edad.csv',
    'umbrales_nacionalid': DATA_DIR / '5.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y nacionalidad.csv',
    'gini':                DATA_DIR / '9.Índice de Gini y Distribución de la renta P80P20.csv',
    'demografico':         DATA_DIR / '10.Indicadores demográficos.csv',
}

# ── Constantes ────────────────────────────────────────────────────────────────
ENCODING  = 'utf-8-sig'
SEPARATOR = ';'
NA_VALUES = ['..', '']

# ── Mapa código INE → nombre provincia ───────────────────────────────────────
COD_PROVINCIA = {
    '01':'Álava',        '02':'Albacete',     '03':'Alicante',    '04':'Almería',
    '05':'Ávila',        '06':'Badajoz',      '07':'Baleares',    '08':'Barcelona',
    '09':'Burgos',       '10':'Cáceres',      '11':'Cádiz',       '12':'Castellón',
    '13':'Ciudad Real',  '14':'Córdoba',      '15':'A Coruña',    '16':'Cuenca',
    '17':'Girona',       '18':'Granada',      '19':'Guadalajara', '20':'Gipuzkoa',
    '21':'Huelva',       '22':'Huesca',       '23':'Jaén',        '24':'León',
    '25':'Lleida',       '26':'La Rioja',     '27':'Lugo',        '28':'Madrid',
    '29':'Málaga',       '30':'Murcia',       '31':'Navarra',     '32':'Ourense',
    '33':'Asturias',     '34':'Palencia',     '35':'Las Palmas',  '36':'Pontevedra',
    '37':'Salamanca',    '38':'S.C. Tenerife','39':'Cantabria',   '40':'Segovia',
    '41':'Sevilla',      '42':'Soria',        '43':'Tarragona',   '44':'Teruel',
    '45':'Toledo',       '46':'Valencia',     '47':'Valladolid',  '48':'Bizkaia',
    '49':'Zamora',       '50':'Zaragoza',     '51':'Ceuta',       '52':'Melilla',
}

# ── Verificación final ────────────────────────────────────────────────────────
print('\n🔍 Verificando ficheros:')
all_ok = True
for nombre, path in FILES.items():
    estado = '✅' if path.exists() else '❌ NO ENCONTRADO'
    if not path.exists(): all_ok = False
    print(f'  {estado}  {nombre:<20} → {path.name[:60]}')

print('\n✅ Configuración lista — todos los ficheros encontrados' if all_ok else
      '\n❌ Revisa los ficheros marcados arriba antes de continuar')

PROJECT_ROOT : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow
DATA_DIR     : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\data\renta_ine
OUTPUT_DIR   : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\output\clean
DATA existe  : True

🔍 Verificando ficheros:
  ✅  renta                → 1.Indicadores de renta media y mediana.csv
  ✅  fuente_ing           → 2.Distribución por fuente de ingresos.csv
  ✅  umbrales_edad        → 4.Porcentaje de población con ingresos por unidad de consumo
  ✅  umbrales_nacionalid  → 5.Porcentaje de población con ingresos por unidad de consumo
  ✅  gini                 → 9.Índice de Gini y Distribución de la renta P80P20.csv
  ✅  demografico          → 10.Indicadores demográficos.csv

✅ Configuración lista — todos los ficheros encontrados


In [9]:
#=====================================================================
# CELDA 2: Carga cruda de los 6 CSV en el diccionario RAW
# (misma lógica que en el exploratorio, base para la limpieza)
#=====================================================================

RAW = {}
for nombre, path in FILES.items():
    RAW[nombre] = pd.read_csv(
        path,
        sep=SEPARATOR,
        encoding=ENCODING,
        na_values=NA_VALUES
    )
    print(f"✅ {nombre:<22} cargado → {RAW[nombre].shape[0]:>10,} filas × {RAW[nombre].shape[1]} columnas")

print(f"\n✅ {len(RAW)} DataFrames cargados en RAW[...]")


✅ renta                  cargado →  3,009,312 filas × 6 columnas
✅ fuente_ing             cargado →  2,507,850 filas × 6 columnas
✅ umbrales_edad          cargado → 13,541,904 filas × 8 columnas
✅ umbrales_nacionalid    cargado →  9,027,936 filas × 8 columnas
✅ gini                   cargado →  1,003,104 filas × 6 columnas
✅ demografico            cargado →  3,510,927 filas × 6 columnas

✅ 6 DataFrames cargados en RAW[...]


In [10]:
#=====================================================================
# CELDA 3: Funciones de limpieza reutilizables (v2 — corregida)
# - parsear_geo()   → separa código INE + nombre limpio
# - limpiar_total() → convierte Total a float/Int64 según contexto
#=====================================================================

def parsear_geo(serie, prefijo):
    """
    Entrada:  '01001 Alegría-Dulantzi'
              '0100101 Alegría-Dulantzi distrito 01'
              '0100101001 Alegría-Dulantzi sección 01001'
    Salida:   cod='01001', nom='Alegría-Dulantzi'
    
    El nombre se extrae quitando el sufijo ' distrito XX' / ' sección XXXXX'
    """
    s = serie.astype(str).str.strip()

    # Código = primer token
    cod = s.str.extract(r'^(\S+)')[0]

    # Nombre = todo tras el primer espacio, sin sufijos geográficos del INE
    nom = (
        s.str.extract(r'^\S+\s+(.+)$')[0]
         .str.replace(r'\s+distrito\s+\d+$',  '', regex=True)
         .str.replace(r'\s+sección\s+\d+$',   '', regex=True)
         .str.strip()
    )

    mask_nan = serie.isna()
    cod[mask_nan] = np.nan
    nom[mask_nan] = np.nan

    return cod.rename(f'cod_{prefijo}'), nom.rename(f'nom_{prefijo}')


def limpiar_total(serie, es_poblacion=False):
    """
    Entrada:  columna Total como str (formato INE español)
    Salida:   (serie_valor, serie_secreto)

    - es_poblacion=True  → entero con puntos de miles → Int64 (nullable)
    - es_poblacion=False → decimal con coma           → float64
    - Punto '.'          → NaN + secreto=True
    """
    s = serie.astype(str).str.strip()

    secreto = (s == '.').rename('secreto_estadistico')

    if es_poblacion:
        s_clean = s.str.replace('.', '', regex=False)
        s_clean = pd.to_numeric(s_clean, errors='coerce').astype('Int64')
    else:
        s_clean = s.str.replace(',', '.', regex=False)
        s_clean = pd.to_numeric(s_clean, errors='coerce')

    return s_clean.rename('valor'), secreto


# ── Tests ────────────────────────────────────────────────────────────
print("🧪 TEST parsear_geo() — municipio, distrito y sección")
test_geo = pd.Series([
    '01001 Alegría-Dulantzi',
    '28079 Madrid',
    np.nan,
    '0100101 Alegría-Dulantzi distrito 01',
    '0100101001 Alegría-Dulantzi sección 01001',
])
cod, nom = parsear_geo(test_geo, 'municipio')
print(pd.DataFrame({'original': test_geo, 'cod': cod, 'nom': nom}).to_string(index=False))

print("\n🧪 TEST limpiar_total() — float (coma decimal)")
test_total = pd.Series(['16.429', '15.116', '.', np.nan, '70,6', '27,4'])
val, sec = limpiar_total(test_total, es_poblacion=False)
print(pd.DataFrame({'original': test_total, 'valor': val, 'secreto': sec}).to_string(index=False))

print("\n🧪 TEST limpiar_total() — Int64 (puntos de miles)")
test_pob = pd.Series(['2.908', '1.613.579', '.', np.nan, '950'])
val, sec = limpiar_total(test_pob, es_poblacion=True)
print(pd.DataFrame({'original': test_pob, 'valor': val, 'secreto': sec}).to_string(index=False))
print(f"  dtype → {val.dtype}")

print("\n✅ Funciones listas")

🧪 TEST parsear_geo() — municipio, distrito y sección
                                 original        cod              nom
                   01001 Alegría-Dulantzi      01001 Alegría-Dulantzi
                             28079 Madrid      28079           Madrid
                                      NaN        NaN              NaN
     0100101 Alegría-Dulantzi distrito 01    0100101 Alegría-Dulantzi
0100101001 Alegría-Dulantzi sección 01001 0100101001 Alegría-Dulantzi

🧪 TEST limpiar_total() — float (coma decimal)
original  valor  secreto
  16.429 16.429    False
  15.116 15.116    False
       .    NaN     True
     NaN    NaN    False
    70,6 70.600    False
    27,4 27.400    False

🧪 TEST limpiar_total() — Int64 (puntos de miles)
 original   valor  secreto
    2.908    2908    False
1.613.579 1613579    False
        .    <NA>     True
      NaN    <NA>    False
      950     950    False
  dtype → Int64

✅ Funciones listas


In [11]:
#=====================================================================
# CELDA 4: Aplicar limpieza a los 6 CSV
# Resultado: diccionario CLEAN con DataFrames normalizados
#=====================================================================

# Columna indicador por CSV (nombre real en cada fichero)
COL_INDICADOR = {
    'renta'              : 'Indicadores de renta media',
    'fuente_ing'         : 'Distribución por fuente de ingresos',
    'umbrales_edad'      : 'Distribución de la renta por unidad de consumo',
    'umbrales_nacionalid': 'Distribución de la renta por unidad de consumo',
    'gini'               : 'Indicadores de renta media',
    'demografico'        : 'Indicadores demográficos',
}

# Columnas extra (dimensiones) por CSV
COLS_EXTRA = {
    'renta'              : [],
    'fuente_ing'         : [],
    'umbrales_edad'      : ['Sexo', 'Tramos de edad'],
    'umbrales_nacionalid': ['Sexo', 'Nacionalidad'],
    'gini'               : [],
    'demografico'        : [],
}

CLEAN = {}

for nombre, df in RAW.items():
    print(f"\n{'─'*60}")
    print(f"  🔧 Limpiando: {nombre.upper()}")
    print(f"{'─'*60}")

    out = pd.DataFrame()

    # ── 1. Geografía ─────────────────────────────────────────────
    cod_mun, nom_mun = parsear_geo(df['Municipios'], 'municipio')
    out['cod_municipio'] = cod_mun
    out['nom_municipio'] = nom_mun

    cod_dis, nom_dis = parsear_geo(df['Distritos'], 'distrito')
    out['cod_distrito'] = cod_dis
    out['nom_distrito'] = nom_dis

    cod_sec, nom_sec = parsear_geo(df['Secciones'], 'seccion')
    out['cod_seccion'] = cod_sec
    out['nom_seccion'] = nom_sec

    # ── 2. Código de provincia (primeros 2 dígitos del municipio) ─
    out['cod_provincia'] = out['cod_municipio'].str[:2]
    out['nom_provincia'] = out['cod_provincia'].map(COD_PROVINCIA)

    # ── 3. Periodo ───────────────────────────────────────────────
    out['periodo'] = df['Periodo'].astype('int16')

    # ── 4. Indicador ─────────────────────────────────────────────
    col_ind = COL_INDICADOR[nombre]
    out['indicador'] = df[col_ind].astype('category')

    # ── 5. Columnas extra (Sexo, Nacionalidad, Tramos de edad) ───
    for col in COLS_EXTRA[nombre]:
        out[col.lower().replace(' ', '_')] = df[col].astype('category')

    # ── 6. Total → valor numérico + flag secreto estadístico ─────
    es_pob = (nombre == 'demografico')  # solo Población tiene enteros
    # Para demografico aplicamos lógica mixta: Población → Int64, resto → float
    if nombre == 'demografico':
        mask_pob = df[col_ind] == 'Población'
        val_pob,  sec_pob  = limpiar_total(df.loc[mask_pob,  'Total'], es_poblacion=True)
        val_rest, sec_rest = limpiar_total(df.loc[~mask_pob, 'Total'], es_poblacion=False)
        # Combinamos en float64 (Int64 y float64 no mezclan bien en una sola columna)
        valor   = pd.concat([val_pob.astype('float64'), val_rest]).reindex(df.index)
        secreto = pd.concat([sec_pob, sec_rest]).reindex(df.index)
    else:
        valor, secreto = limpiar_total(df['Total'], es_poblacion=False)

    out['valor']                = valor
    out['secreto_estadistico']  = secreto.astype(bool)

    CLEAN[nombre] = out

    # ── Verificación rápida ──────────────────────────────────────
    pct_ok  = out['valor'].notna().sum() / len(out) * 100
    pct_sec = out['secreto_estadistico'].sum() / len(out) * 100
    print(f"  Shape    : {out.shape[0]:>10,} filas × {out.shape[1]} columnas")
    print(f"  Columnas : {list(out.columns)}")
    print(f"  valor OK : {pct_ok:>6.1f}%  |  secreto estadístico: {pct_sec:.1f}%")
    print(f"  dtypes   :")
    for col, dtype in out.dtypes.items():
        print(f"    {col:<30} {str(dtype)}")

print(f"\n\n✅ Limpieza completa — {len(CLEAN)} DataFrames en CLEAN[...]")



────────────────────────────────────────────────────────────
  🔧 Limpiando: RENTA
────────────────────────────────────────────────────────────
  Shape    :  3,009,312 filas × 12 columnas
  Columnas : ['cod_municipio', 'nom_municipio', 'cod_distrito', 'nom_distrito', 'cod_seccion', 'nom_seccion', 'cod_provincia', 'nom_provincia', 'periodo', 'indicador', 'valor', 'secreto_estadistico']
  valor OK :   91.8%  |  secreto estadístico: 5.4%
  dtypes   :
    cod_municipio                  str
    nom_municipio                  str
    cod_distrito                   str
    nom_distrito                   str
    cod_seccion                    str
    nom_seccion                    str
    cod_provincia                  str
    nom_provincia                  str
    periodo                        int16
    indicador                      category
    valor                          float64
    secreto_estadistico            bool

────────────────────────────────────────────────────────────
  🔧 Li

In [12]:
#=====================================================================
# CELDA 5: Verificación post-limpieza — muestra de filas reales
# para confirmar visualmente que los datos tienen buen aspecto
#=====================================================================

for nombre, df in CLEAN.items():
    print(f"\n{'='*70}")
    print(f"  📄 {nombre.upper()} — primeras 3 filas nivel municipio")
    print(f"{'='*70}")
    # Solo nivel municipio (sin distrito ni sección) para mayor claridad
    muestra = df[df['cod_distrito'].isna()].head(3)
    print(muestra.to_string(index=False))

print("\n" + "="*70)
print("📊 RESUMEN FINAL DE CLEAN")
print("="*70)

resumen = []
for nombre, df in CLEAN.items():
    resumen.append({
        'CSV'                   : nombre,
        'Filas'                 : f"{len(df):,}",
        'Columnas'              : df.shape[1],
        'valor OK %'            : f"{df['valor'].notna().sum()/len(df)*100:.1f}%",
        'Secreto estadístico %' : f"{df['secreto_estadistico'].sum()/len(df)*100:.1f}%",
        'NaN puro %'            : f"{(df['valor'].isna() & ~df['secreto_estadistico']).sum()/len(df)*100:.1f}%",
    })

print(pd.DataFrame(resumen).set_index('CSV').to_string())
print("\n✅ DataFrames listos para guardar")


  📄 RENTA — primeras 3 filas nivel municipio
cod_municipio    nom_municipio cod_distrito nom_distrito cod_seccion nom_seccion cod_provincia nom_provincia  periodo                    indicador  valor  secreto_estadistico
        01001 Alegría-Dulantzi          NaN          NaN         NaN         NaN            01         Álava     2023 Renta neta media por persona 16.429                False
        01001 Alegría-Dulantzi          NaN          NaN         NaN         NaN            01         Álava     2022 Renta neta media por persona 15.116                False
        01001 Alegría-Dulantzi          NaN          NaN         NaN         NaN            01         Álava     2021 Renta neta media por persona 14.647                False

  📄 FUENTE_ING — primeras 3 filas nivel municipio
cod_municipio    nom_municipio cod_distrito nom_distrito cod_seccion nom_seccion cod_provincia nom_provincia  periodo                  indicador  valor  secreto_estadistico
        01001 Alegría-Dulantzi

In [13]:
#=====================================================================
# CELDA 6: Nulos por columna en CLEAN — decidir qué columnas
# eliminar antes de guardar para optimizar espacio en disco
#=====================================================================

print("=" * 75)
print("🔍 NULOS POR COLUMNA EN CADA DATAFRAME LIMPIO")
print("=" * 75)

for nombre, df in CLEAN.items():
    total = len(df)
    print(f"\n{'─'*75}")
    print(f"  📄 {nombre.upper()}  —  {total:,} filas × {df.shape[1]} columnas")
    print(f"{'─'*75}")
    for col in df.columns:
        nulos = df[col].isna().sum()
        pct   = nulos / total * 100
        barra = '█' * int(pct / 5)  # barra visual cada 5%
        print(f"  {col:<30} {nulos:>10,}  ({pct:>6.1f}%)  {barra}")

print("\n" + "=" * 75)
print("📦 TAMAÑO ESTIMADO EN MEMORIA POR DATAFRAME")
print("=" * 75)
for nombre, df in CLEAN.items():
    mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"  {nombre:<22} → {mb:>8.1f} MB  ({df.shape[0]:>10,} filas × {df.shape[1]} cols)")
    

🔍 NULOS POR COLUMNA EN CADA DATAFRAME LIMPIO

───────────────────────────────────────────────────────────────────────────
  📄 RENTA  —  3,009,312 filas × 12 columnas
───────────────────────────────────────────────────────────────────────────
  cod_municipio                           0  (   0.0%)  
  nom_municipio                           0  (   0.0%)  
  cod_distrito                      439,506  (  14.6%)  ██
  nom_distrito                      439,506  (  14.6%)  ██
  cod_seccion                     1,007,424  (  33.5%)  ██████
  nom_seccion                     1,007,424  (  33.5%)  ██████
  cod_provincia                           0  (   0.0%)  
  nom_provincia                           0  (   0.0%)  
  periodo                                 0  (   0.0%)  
  indicador                               0  (   0.0%)  
  valor                             247,560  (   8.2%)  █
  secreto_estadistico                     0  (   0.0%)  

────────────────────────────────────────────────────────

In [14]:
#=====================================================================
# CELDA 7: Diagnóstico de NaN reales en valor — separando por
# nivel geográfico para saber dónde están los datos de verdad
#=====================================================================

print("=" * 75)
print("🔍 NULOS EN 'valor' DESGLOSADOS POR NIVEL GEOGRÁFICO")
print("=" * 75)

for nombre, df in CLEAN.items():
    total = len(df)
    
    # Separamos los tres niveles
    mask_mun  = df['cod_distrito'].isna() & df['cod_seccion'].isna()
    mask_dist = df['cod_distrito'].notna() & df['cod_seccion'].isna()
    mask_sec  = df['cod_seccion'].notna()

    print(f"\n{'─'*75}")
    print(f"  📄 {nombre.upper()}")
    print(f"{'─'*75}")
    print(f"  {'Nivel':<12} {'Filas':>10} {'NaN valor':>12} {'Secreto .':>12} {'Valor OK':>12}")
    print(f"  {'─'*60}")

    for label, mask in [('municipio', mask_mun), ('distrito', mask_dist), ('seccion', mask_sec)]:
        sub        = df[mask]
        n          = len(sub)
        nan_real   = (sub['valor'].isna() & ~sub['secreto_estadistico']).sum()
        secreto    = sub['secreto_estadistico'].sum()
        ok         = sub['valor'].notna().sum()
        pct_ok     = ok / n * 100 if n > 0 else 0
        pct_nan    = nan_real / n * 100 if n > 0 else 0
        pct_sec    = secreto / n * 100 if n > 0 else 0
        print(f"  {label:<12} {n:>10,} {nan_real:>10,} ({pct_nan:>4.1f}%) "
              f"{secreto:>10,} ({pct_sec:>4.1f}%) "
              f"{ok:>10,} ({pct_ok:>4.1f}%)")
        

🔍 NULOS EN 'valor' DESGLOSADOS POR NIVEL GEOGRÁFICO



───────────────────────────────────────────────────────────────────────────
  📄 RENTA
───────────────────────────────────────────────────────────────────────────
  Nivel             Filas    NaN valor    Secreto .     Valor OK
  ────────────────────────────────────────────────────────────
  municipio       439,506      8,628 ( 2.0%)     51,784 (11.8%)    379,094 (86.3%)
  distrito        567,918     12,148 ( 2.1%)     53,004 ( 9.3%)    502,766 (88.5%)
  seccion       2,001,888     64,652 ( 3.2%)     57,344 ( 2.9%)  1,879,892 (93.9%)

───────────────────────────────────────────────────────────────────────────
  📄 FUENTE_ING
───────────────────────────────────────────────────────────────────────────
  Nivel             Filas    NaN valor    Secreto .     Valor OK
  ────────────────────────────────────────────────────────────
  municipio       366,255     10,656 ( 2.9%)     63,910 (17.4%)    291,689 (79.6%)
  distrito        473,265     15,108 ( 3.2%)     65,250 (13.8%)    392,907 (83.0%

In [15]:
#=====================================================================
# CELDA 8: Optimización — eliminar nivel distrito y columnas
# redundantes antes de guardar en disco
# Eliminamos: filas distrito + cod_distrito + nom_distrito + nom_seccion
#=====================================================================

COLS_ELIMINAR = ['cod_distrito', 'nom_distrito', 'nom_seccion']

CLEAN_OPT = {}

print("=" * 75)
print("✂️  OPTIMIZACIÓN DE DATAFRAMES")
print("=" * 75)

for nombre, df in CLEAN.items():

    # ── 1. Eliminar filas de nivel distrito ──────────────────────
    mask_no_distrito = ~(df['cod_distrito'].notna() & df['cod_seccion'].isna())
    df_opt = df[mask_no_distrito].copy()

    # ── 2. Eliminar columnas redundantes ─────────────────────────
    df_opt = df_opt.drop(columns=COLS_ELIMINAR)

    # ── 3. Reset index ───────────────────────────────────────────
    df_opt = df_opt.reset_index(drop=True)

    CLEAN_OPT[nombre] = df_opt

    # ── Reporte ──────────────────────────────────────────────────
    filas_antes  = len(df)
    filas_dopo   = len(df_opt)
    filas_elim   = filas_antes - filas_dopo
    cols_antes   = df.shape[1]
    cols_dopo    = df_opt.shape[1]
    mb_antes     = df.memory_usage(deep=True).sum() / (1024**2)
    mb_dopo      = df_opt.memory_usage(deep=True).sum() / (1024**2)
    ahorro_pct   = (1 - mb_dopo / mb_antes) * 100

    print(f"\n  📄 {nombre.upper()}")
    print(f"    Filas  : {filas_antes:>10,} → {filas_dopo:>10,}  (-{filas_elim:,} filas distrito)")
    print(f"    Cols   : {cols_antes:>5} → {cols_dopo:>5}  (-{cols_antes - cols_dopo} columnas)")
    print(f"    Memoria: {mb_antes:>8.1f} MB → {mb_dopo:>8.1f} MB  (ahorro: {ahorro_pct:.1f}%)")
    print(f"    Columnas restantes: {list(df_opt.columns)}")

print("\n" + "=" * 75)
print("📦 RESUMEN TOTAL DE AHORRO")
print("=" * 75)
mb_total_antes = sum(df.memory_usage(deep=True).sum() for df in CLEAN.values()) / (1024**2)
mb_total_dopo  = sum(df.memory_usage(deep=True).sum() for df in CLEAN_OPT.values()) / (1024**2)
print(f"  Memoria total antes : {mb_total_antes:>8.1f} MB  ({mb_total_antes/1024:.2f} GB)")
print(f"  Memoria total después: {mb_total_dopo:>8.1f} MB  ({mb_total_dopo/1024:.2f} GB)")
print(f"  Ahorro total         : {mb_total_antes - mb_total_dopo:>8.1f} MB  ({(1 - mb_total_dopo/mb_total_antes)*100:.1f}%)")

print("\n✅ CLEAN_OPT listo — pendiente de verificación antes de guardar")

✂️  OPTIMIZACIÓN DE DATAFRAMES

  📄 RENTA
    Filas  :  3,009,312 →  2,441,394  (-567,918 filas distrito)
    Cols   :    12 →     9  (-3 columnas)
    Memoria:    379.8 MB →    201.7 MB  (ahorro: 46.9%)
    Columnas restantes: ['cod_municipio', 'nom_municipio', 'cod_seccion', 'cod_provincia', 'nom_provincia', 'periodo', 'indicador', 'valor', 'secreto_estadistico']

  📄 FUENTE_ING
    Filas  :  2,507,850 →  2,034,585  (-473,265 filas distrito)
    Cols   :    12 →     9  (-3 columnas)
    Memoria:    316.5 MB →    168.1 MB  (ahorro: 46.9%)
    Columnas restantes: ['cod_municipio', 'nom_municipio', 'cod_seccion', 'cod_provincia', 'nom_provincia', 'periodo', 'indicador', 'valor', 'secreto_estadistico']

  📄 UMBRALES_EDAD
    Filas  : 13,541,904 → 10,986,273  (-2,555,631 filas distrito)
    Cols   :    14 →    11  (-3 columnas)
    Memoria:   1735.0 MB →    928.6 MB  (ahorro: 46.5%)
    Columnas restantes: ['cod_municipio', 'nom_municipio', 'cod_seccion', 'cod_provincia', 'nom_provincia',

In [16]:
#=====================================================================
# CELDA 9: Verificación final de CLEAN_OPT antes de guardar
# Muestra una muestra de municipio y una de sección por CSV
#=====================================================================

print("=" * 75)
print("🔍 VERIFICACIÓN FINAL — muestra municipio y sección por CSV")
print("=" * 75)

for nombre, df in CLEAN_OPT.items():
    print(f"\n{'─'*75}")
    print(f"  📄 {nombre.upper()}")
    print(f"{'─'*75}")

    # Nivel municipio (cod_seccion es NaN)
    mun = df[df['cod_seccion'].isna()].head(2)
    print(f"  🏘️  Nivel MUNICIPIO:")
    print(mun.to_string(index=False))

    # Nivel sección (cod_seccion no es NaN)
    sec = df[df['cod_seccion'].notna()].head(2)
    print(f"\n  🗺️  Nivel SECCIÓN:")
    print(sec.to_string(index=False))

print("\n" + "=" * 75)
print("📊 SHAPE FINAL POR CSV")
print("=" * 75)
for nombre, df in CLEAN_OPT.items():
    mun = df['cod_seccion'].isna().sum()
    sec = df['cod_seccion'].notna().sum()
    print(f"  {nombre:<22} → {len(df):>10,} filas  |  municipio: {mun:>8,}  |  sección: {sec:>9,}")

print("\n✅ Todo OK — listos para guardar en disco")

🔍 VERIFICACIÓN FINAL — muestra municipio y sección por CSV

───────────────────────────────────────────────────────────────────────────
  📄 RENTA
───────────────────────────────────────────────────────────────────────────
  🏘️  Nivel MUNICIPIO:
cod_municipio    nom_municipio cod_seccion cod_provincia nom_provincia  periodo                    indicador  valor  secreto_estadistico
        01001 Alegría-Dulantzi         NaN            01         Álava     2023 Renta neta media por persona 16.429                False
        01001 Alegría-Dulantzi         NaN            01         Álava     2022 Renta neta media por persona 15.116                False

  🗺️  Nivel SECCIÓN:
cod_municipio    nom_municipio cod_seccion cod_provincia nom_provincia  periodo                    indicador  valor  secreto_estadistico
        01001 Alegría-Dulantzi  0100101001            01         Álava     2023 Renta neta media por persona 17.069                False
        01001 Alegría-Dulantzi  0100101001      

In [17]:
#=====================================================================
# CELDA 10: Guardar los 6 DataFrames optimizados en output/clean/
# Formato: CSV separador ';' y encoding utf-8-sig
# Separamos en dos ficheros por CSV: _municipio y _seccion
#=====================================================================

NOMBRES_SALIDA = {
    'renta'              : 'renta',
    'fuente_ing'         : 'fuente_ingresos',
    'umbrales_edad'      : 'umbrales_edad',
    'umbrales_nacionalid': 'umbrales_nacionalidad',
    'gini'               : 'gini',
    'demografico'        : 'demografico',
}

print("💾 Guardando ficheros en:")
print(f"   {OUTPUT_DIR}\n")

resumen_disco = []

for nombre, df in CLEAN_OPT.items():
    base = NOMBRES_SALIDA[nombre]

    # ── Nivel municipio ───────────────────────────────────────────
    df_mun = df[df['cod_seccion'].isna()].drop(columns=['cod_seccion']).reset_index(drop=True)
    ruta_mun = OUTPUT_DIR / f'{base}_municipio.csv'
    df_mun.to_csv(ruta_mun, sep=';', encoding='utf-8-sig', index=False)
    mb_mun = ruta_mun.stat().st_size / (1024**2)

    # ── Nivel sección ─────────────────────────────────────────────
    df_sec = df[df['cod_seccion'].notna()].reset_index(drop=True)
    ruta_sec = OUTPUT_DIR / f'{base}_seccion.csv'
    df_sec.to_csv(ruta_sec, sep=';', encoding='utf-8-sig', index=False)
    mb_sec = ruta_sec.stat().st_size / (1024**2)

    print(f"  ✅ {base}")
    print(f"       _municipio.csv  →  {len(df_mun):>8,} filas  |  {mb_mun:>7.1f} MB")
    print(f"       _seccion.csv    →  {len(df_sec):>8,} filas  |  {mb_sec:>7.1f} MB")

    resumen_disco.append({
        'CSV'           : base,
        'Filas mun'     : len(df_mun),
        'MB mun'        : round(mb_mun, 1),
        'Filas sec'     : len(df_sec),
        'MB sec'        : round(mb_sec, 1),
        'MB total'      : round(mb_mun + mb_sec, 1),
    })

print("\n" + "=" * 75)
print("📊 RESUMEN FINAL EN DISCO")
print("=" * 75)
df_resumen = pd.DataFrame(resumen_disco).set_index('CSV')
print(df_resumen.to_string())
mb_total = df_resumen['MB total'].sum()
print(f"\n  💾 Tamaño total en disco: {mb_total:.1f} MB  ({mb_total/1024:.2f} GB)")

print("\n" + "=" * 75)
print("🔍 Verificación — todos los ficheros existen en disco")
print("=" * 75)
all_ok = True
for nombre in NOMBRES_SALIDA.values():
    for sufijo in ['municipio', 'seccion']:
        ruta = OUTPUT_DIR / f'{nombre}_{sufijo}.csv'
        estado = '✅' if ruta.exists() else '❌'
        if not ruta.exists(): all_ok = False
        print(f"  {estado}  {nombre}_{sufijo}.csv")

print(f"\n{'✅ Todos los ficheros guardados correctamente' if all_ok else '❌ Revisa los ficheros marcados'}")

💾 Guardando ficheros en:
   c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\output\clean

  ✅ renta
       _municipio.csv  →   439,506 filas  |     34.1 MB
       _seccion.csv    →  2,001,888 filas  |    174.7 MB
  ✅ fuente_ingresos
       _municipio.csv  →   366,255 filas  |     28.5 MB
       _seccion.csv    →  1,668,330 filas  |    145.5 MB
  ✅ umbrales_edad
       _municipio.csv  →  1,977,777 filas  |    265.6 MB
       _seccion.csv    →  9,008,496 filas  |   1283.5 MB
  ✅ umbrales_nacionalidad
       _municipio.csv  →  1,318,518 filas  |    168.0 MB
       _seccion.csv    →  6,005,664 filas  |    817.6 MB
  ✅ gini
       _municipio.csv  →   146,502 filas  |     10.0 MB
       _seccion.csv    →   667,296 filas  |     52.0 MB
  ✅ demografico
       _municipio.csv  →   512,757 filas  |     38.7 MB
       _seccion.csv    →  2,335,536 filas  |    198.2 MB

📊 RESUMEN FINAL EN DISCO
                       Filas mun  MB mun  Filas sec  MB sec  MB total
CSV   